# Maze Crawler — Fallback Agent (fb3): Eval & Submit

Robust **survivor**: BFS north-pathing over the remembered wall map, jump-to-escape
when boxed in, strict own-half containment (so it can never ram the enemy factory),
and a small scout screen for vision that also banks crystal energy for the step-500
energy tiebreak.

**No GPU needed.** Run top-to-bottom. The `%%writefile` cell creates `main.py` for submission.


In [ ]:
import kaggle_environments
from kaggle_environments import make
v = getattr(kaggle_environments, '__version__', '?')
print('kaggle_environments:', v)
env = make('crawl', configuration={'randomSeed': 42})
cfg = dict(env.configuration)
print('episodeSteps:', cfg['episodeSteps'])
print('scroll interval:', cfg['scrollStartInterval'], '->', cfg['scrollEndInterval'], '(ramp by step', str(cfg['scrollRampSteps'])+')')
print('factory move period:', cfg['factoryMovePeriod'], '| jump cd:', cfg['factoryJumpCooldown'])


In [ ]:
%%writefile main.py
"""Maze Crawler FALLBACK agent. VERSION=fb3.

Strategy: never fall behind the scroll (BFS climb on the remembered wall map,
jump over walls when blocked, stay strictly on our own half so we can never ram
the enemy factory). A small non-colliding scout screen rushes north to reveal
the maze ahead (walls are remembered forever) so the factory routes around
dead-ends instead of backtracking blind. Scouts auto-collect crystals; their
energy counts toward the step-500 energy tiebreak.
"""
from collections import deque

VERSION = "fb3"
FACTORY, SCOUT, WORKER, MINER = 0, 1, 2, 3
WN, WE, WS, WW = 1, 2, 4, 8
DIRS = [("NORTH", 0, 1, WN), ("EAST", 1, 0, WE), ("WEST", -1, 0, WW), ("SOUTH", 0, -1, WS)]
DIR_OFF = {"NORTH": (0, 1), "EAST": (1, 0), "WEST": (-1, 0), "SOUTH": (0, -1)}
DIR_BIT = {"NORTH": WN, "EAST": WE, "WEST": WW, "SOUTH": WS}
SAFE_GAP = 3
TARGET_SCOUTS = 2


def agent(obs, config):
    try:
        return _agent(obs, config)
    except Exception:
        out = {}
        try:
            for uid, d in obs.robots.items():
                if d[4] == obs.player:
                    out[uid] = "IDLE"  # safe: never walk off-board on error
        except Exception:
            pass
        return out


def _agent(obs, config):
    width = config.width
    half = width // 2
    south = obs.southBound
    north = obs.northBound
    walls = obs.walls
    nwalls = len(walls)
    player = obs.player
    left_is_mine = (player == 0)
    my_center = (half // 2) if left_is_mine else (half + half // 2)
    lo, hi = (0, half - 1) if left_is_mine else (half, width - 1)

    def in_my_half(col):
        return (col < half) if left_is_mine else (col >= half)

    def wat(c, r):
        if c < 0 or c >= width or r < south or r > north:
            return -1
        i = (r - south) * width + c
        return walls[i] if 0 <= i < nwalls else -1

    def step_target(c, r, name):
        w = wat(c, r)
        if w == -1 or (w & DIR_BIT[name]):
            return None
        dc, dr = DIR_OFF[name]
        nc, nr = c + dc, r + dr
        if nc < 0 or nc >= width or nr < south or nr > north or not in_my_half(nc):
            return None
        return (nc, nr)

    def plan(sc, sr, avoid_south, bias):
        dist = {(sc, sr): 0}
        par = {}
        q = deque([(sc, sr)])
        best = (sc, sr)
        bestk = (sr, 0, -abs(sc - bias))
        while q:
            c, r = q.popleft()
            w = wat(c, r)
            if w == -1:
                continue
            for name, dc, dr, bit in DIRS:
                if w & bit:
                    continue
                nc, nr = c + dc, r + dr
                if nc < 0 or nc >= width or nr < south or nr > north or not in_my_half(nc):
                    continue
                if avoid_south and name == "SOUTH" and (nr - south) < SAFE_GAP:
                    continue
                if (nc, nr) in dist:
                    continue
                dist[(nc, nr)] = dist[(c, r)] + 1
                par[(nc, nr)] = (c, r, name)
                k = (nr, -dist[(nc, nr)], -abs(nc - bias))
                if k > bestk:
                    bestk = k
                    best = (nc, nr)
                q.append((nc, nr))
        if best == (sc, sr):
            return None, best[1]
        cur = best
        first = None
        while cur in par:
            pc, pr, name = par[cur]
            first = name
            cur = (pc, pr)
        return first, best[1]

    my = [(uid, list(d)) for uid, d in obs.robots.items() if d[4] == player]
    counts = {}
    for _, d in my:
        counts[d[0]] = counts.get(d[0], 0) + 1
    factory = [(u, d) for u, d in my if d[0] == FACTORY]
    others = sorted([(u, d) for u, d in my if d[0] != FACTORY], key=lambda x: x[0])

    occupied_end = set()
    remaining = {(d[1], d[2]) for _, d in others}
    actions = {}

    # ---- Factory first (it crushes friendlies, so it ignores their cells) ----
    for uid, d in factory:
        col, row, energy = d[1], d[2], d[3]
        move_cd, jump_cd, build_cd = d[5], d[6], d[7]
        gap = row - south
        if (counts.get(SCOUT, 0) < TARGET_SCOUTS and build_cd == 0
                and energy >= config.scoutCost + 250 and gap >= 4):
            done = False
            for sd in ("NORTH", "EAST", "WEST"):
                t = step_target(col, row, sd)
                if t and t not in occupied_end:
                    actions[uid] = "BUILD_SCOUT" if sd == "NORTH" else "BUILD_SCOUT_" + sd
                    occupied_end.add(t)
                    occupied_end.add((col, row))
                    done = True
                    break
            if done:
                continue
        avoid_south = gap < SAFE_GAP
        st, reach = plan(col, row, avoid_south, my_center)
        chosen, tgt = None, (col, row)
        if st is not None and reach > row:
            t = step_target(col, row, st)
            if t:
                chosen, tgt = st, t
        if chosen is None and jump_cd == 0 and move_cd == 0 and row + 2 <= north:
            chosen, tgt = "JUMP_NORTH", (col, row + 2)
        if chosen is None and st is not None:
            t = step_target(col, row, st)
            if t:
                chosen, tgt = st, t
        if chosen is None:
            chosen = "IDLE"
        actions[uid] = chosen
        occupied_end.add(tgt)

    # ---- Other units: collision-free climb ----
    for uid, d in others:
        col, row = d[1], d[2]
        remaining.discard((col, row))
        forbidden = occupied_end | remaining
        gap = row - south
        avoid_south = gap < SAFE_GAP
        bias = my_center + (hash(uid) % 5 - 2)
        bias = min(max(bias, lo), hi)
        st, reach = plan(col, row, avoid_south, bias)
        ranked = []
        if st:
            ranked.append(st)
        for name in ("NORTH", "EAST", "WEST", "SOUTH"):
            if name not in ranked:
                ranked.append(name)
        chosen, tgt = None, (col, row)
        for name in ranked:
            if avoid_south and name == "SOUTH":
                continue
            t = step_target(col, row, name)
            if t and t not in forbidden:
                chosen, tgt = name, t
                break
        if chosen is None:
            chosen = "IDLE"
        actions[uid] = chosen
        occupied_end.add(tgt)

    return actions


In [ ]:
# Sample game vs the built-in random agent, then render it.
env = make('crawl', configuration={'randomSeed': 7})
env.run(['main.py', 'random'])
last = env.steps[-1]
print('episode length:', len(env.steps), '| rewards:', [s.reward for s in last])
env.render(mode='ipython', width=700, height=700)


In [ ]:
# Survival + win-rate over several seeds vs random and a naive 'march north' bot.
import statistics
def march(obs, config):
    a = {}
    for uid, d in obs.robots.items():
        if d[4] == obs.player:
            a[uid] = 'BUILD_WORKER' if d[0] == 0 else 'NORTH'
    return a

for name, opp in [('random', 'random'), ('march', march)]:
    w = l = t = 0; lens = []
    for seed in range(1, 11):
        env = make('crawl', configuration={'randomSeed': seed})
        env.run(['main.py', opp])
        r0, r1 = [s.reward for s in env.steps[-1]]
        w += r0 > r1; l += r0 < r1; t += r0 == r1; lens.append(len(env.steps))
    print(f'vs {name:6s}:  W/L/T = {w}/{l}/{t}   avg_episode_len = {statistics.mean(lens):.0f}')


## Submit

This notebook wrote `main.py` to the working directory. Submit it as a single-file agent:

```bash
kaggle competitions submit maze-crawler -f main.py -m "fb3 survivor"
kaggle competitions submissions maze-crawler
```

Or **Save Version** of this notebook and use the *Submit to Competition* button (the saved
output must contain `main.py`). Remember: only your latest 2 submissions count toward the
final rating, so keep a strong one in slot.
